# learning-to-focus: colab runner

Runtime -> Change runtime type -> **A100** for real runs (T4 is fine for the smoke test).

Everything that matters gets written to Google Drive under `MyDrive/learning-to-focus/`, so if the
session dies you just rerun the setup cells and then the same experiment cell. Generation and both
trainers resume from where they stopped.

Run the setup cells top to bottom, then **only the experiment cell you want**. Rough timings are
in each cell and they're guesses until we've actually run them once.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/learning-to-focus/{runs,checkpoints,data}

In [ ]:
# fresh clone every session (or pull if it's already there)
%cd /content
!test -d learning-to-focus && (cd learning-to-focus && git pull -q) || git clone -q https://github.com/Adi-Kaul/learning-to-focus.git
%cd /content/learning-to-focus
!git log --oneline | head -3

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# optional: weights & biases. paste your key when asked (wandb.ai/authorize)
import wandb
wandb.login()

In [ ]:
# data. the RULER parquet files are ~100MB total and cached by HF, the split is deterministic
# and checked against data/splits/niah_test_ids.txt
!python -m src.data --test-per-bucket 20
!diff <(python -c "import json;[print(json.loads(l)['id']) for l in open('data/niah_test.jsonl')]") data/splits/niah_test_ids.txt && echo "test split matches the committed one"
!python -m src.synth

In [ ]:
# unit tests, ~2-3 min (the decode equivalence test runs qwen3-0.6b)
!python -m pytest -q tests/

## 0. smoke test (~2 min, any GPU)
Make sure the whole thing runs on GPU before spending real time.

In [ ]:
!python -m src.generate --config configs/smoke_niah.yaml --out-dir /content/drive/MyDrive/learning-to-focus/runs

## 1. warm-up: Qwen3-0.6B on fake documents (phase 1)
before -> SFT -> after on 200 synth test questions. ~10 min eval each, SFT ~15-20 min on A100.

In [ ]:
R = "/content/drive/MyDrive/learning-to-focus"
!python -m src.generate --model Qwen/Qwen3-0.6B --data data/synth_test.jsonl --arms da_nomask da --max-new-tokens 256 --batch-size 1 --out-dir {R}/runs --run-name synth_0.6b_before
!python -m src.train_sft --config configs/sft_synth_0.6b.yaml
!python -m src.generate --model Qwen/Qwen3-0.6B --adapter {R}/checkpoints/sft_synth_0.6b/final --data data/synth_test.jsonl --arms da_nomask da --max-new-tokens 256 --batch-size 1 --out-dir {R}/runs --run-name synth_0.6b_after

## 2. baseline: Qwen3-4B, vanilla vs DA-no-mask vs DA (phase 2)
297 test questions x 3 arms, batch size 1. My guess is **2-4 hours** on an A100 (16K prompts are the slow part).
Resumable, so it's fine to do it over two sessions. Try `--buckets 4k` first if you want a fast early look (~30-45 min).

In [ ]:
!python -m src.generate --config configs/baseline_qwen3_4b.yaml

## 3. SFT data + SFT (phase 3)
Rejection sampling: 600 training questions x 8 samples under the mask. **Expensive**, maybe 3-5 hours,
and the acceptance rate decides everything. Run 50 questions first and look at how many get kept.
If it's close to zero, use the `--ideal` fallback (templated traces, instant) and say so in the write-up.

In [ ]:
!python -m src.sample_traces --model Qwen/Qwen3-4B --n-questions 50 --k 8 --buckets 4k 8k --out /content/drive/MyDrive/learning-to-focus/data/sft_kept.jsonl

In [ ]:
# SFT on whatever got kept. ~1-2 hours for ~1000 traces at 4-8K tokens
!python -m src.train_sft --config configs/sft_qwen3_4b.yaml
!python -m src.generate --config configs/eval_sft_qwen3_4b.yaml --run-name eval_sft_qwen3_4b

## 4. GRPO (phase 4)
First the vanilla reference costs for the training questions (~30 min for 400 questions at 4K),
then a **pilot** (20 steps, ~30-60 min), read the logged completions for reward hacking, then the real run.

In [ ]:
!python -m src.sample_traces --vanilla-ref --model Qwen/Qwen3-4B --n-questions 400 --buckets 4k --out /content/drive/MyDrive/learning-to-focus/data/grpo_train.jsonl

In [ ]:
!python -m src.train_grpo --config configs/grpo_qwen3_4b.yaml --max-steps 20 --out-dir /content/drive/MyDrive/learning-to-focus/checkpoints/grpo_pilot

In [ ]:
!python -m src.train_grpo --config configs/grpo_qwen3_4b.yaml
!python -m src.generate --config configs/eval_grpo_qwen3_4b.yaml --run-name eval_grpo_qwen3_4b

## copy small results back
summary/metrics CSVs are small, traces are not. Download these and drop them in `results/` in the repo.

In [ ]:
!cd /content/drive/MyDrive/learning-to-focus/runs && zip -qr /content/results_small.zip */summary.csv */metrics.csv */meta.json && ls -la /content/results_small.zip
from google.colab import files
files.download('/content/results_small.zip')